# combined vega and hsbd databases

In [1]:
import sys
from pathlib import Path

notebookdir = Path.cwd().parent
sys.path.append(str(notebookdir))  # this way we can import src modules even in different subfolders

from typing import List
import pandas as pd
import numpy as np

import sqlalchemy as sa
from sqlalchemy.orm import sessionmaker
from src.db_schema import *
from src.db_utils import *
from src.rdkit_tools import smiles_standardizer_msc

In [2]:
# set directories and filenames
working_dir = Path.cwd().parent
data_dir = working_dir / "processed_data"

# vega source db (read-only source)
db_vega = data_dir / "vega_t_half_soil_water_sediment.db"
engine_vega = sa.create_engine(f"sqlite:///{db_vega}")
Session_veg = sessionmaker(bind=engine_vega)

# hsbd source db (read-only source)
db_hsbd = data_dir / "hsbd_t_half_all.db"
engine_hsbd = sa.create_engine(f"sqlite:///{db_hsbd}")
Session_hsbd = sessionmaker(bind=engine_hsbd)

# combined db: create schema using Base (same structure as hsbd)
db_combined = data_dir / "combined_t_half_vega_hsbd_soil_water_sediment.db"
engine_combined = sa.create_engine(f"sqlite:///{db_combined}")
Session_combined = sessionmaker(bind=engine_combined)
if not db_combined.exists():
    Base.metadata.create_all(
        engine_combined,
        tables=[
            SoilData.__table__,
            WaterData.__table__,
            SedimentData.__table__,
        ],
    )
    print(f"Created combined db at: {db_combined}")
else:
    # drop all existing data to avoid duplicates when re-running the script
    with Session_combined() as session:
        session.query(SoilData).delete()
        session.query(WaterData).delete()
        session.query(SedimentData).delete()
        session.commit()
    print(f"Cleared existing data from combined db: {db_combined}")

Created combined db at: /Users/a/dev/WP3bioddeg_V2/t_half/processed_data/combined_t_half_vega_hsbd_soil_water_sediment.db


In [3]:
# create_all uses CREATE TABLE IF NOT EXISTS — safe to re-run, will not overwrite existing tables
# verify db_combined schema: check all tables exist and are empty
session_combined = Session_combined()

table_classes = {
    "soil_data": SoilData,
    "water_data": WaterData,
    "sediment_data": SedimentData,
}

for table_name, table_cls in table_classes.items():
    n_rows = session_combined.query(table_cls).count()
    n_cols = len(table_cls.__table__.columns)
    print(f"{table_name}: {n_rows} rows, {n_cols} columns")

session_combined.close()
print("Done — db_combined is empty and ready for data.")

soil_data: 0 rows, 379 columns
water_data: 0 rows, 379 columns
sediment_data: 0 rows, 379 columns
Done — db_combined is empty and ready for data.


In [4]:
# read data from source dbs into pandas DataFrames for later combining and inserting into combined db
soil_vega = get_all_data("soil", Session_veg)
water_vega = get_all_data("water", Session_veg)
sediment_vega = get_all_data("sediment", Session_veg)
# add "_vega" suffix to "reference" values
soil_vega["reference"] = soil_vega["reference"].astype(str) + "_vega"
water_vega["reference"] = water_vega["reference"].astype(str) + "_vega"
sediment_vega["reference"] = sediment_vega["reference"].astype(str) + "_vega"

soil_hsbd = get_all_data("soil", Session_hsbd)
water_hsbd = get_all_data("water", Session_hsbd)
sediment_hsbd = get_all_data("sediment", Session_hsbd)
# add "_hsbd" suffix to "reference" values
soil_hsbd["reference"] = soil_hsbd["reference"].astype(str) + "_hsbd"
water_hsbd["reference"] = water_hsbd["reference"].astype(str) + "_hsbd"
sediment_hsbd["reference"] = sediment_hsbd["reference"].astype(str) + "_hsbd"

In [5]:
soil_combined = pd.concat([soil_hsbd, soil_vega], ignore_index=True)
water_combined = pd.concat([water_hsbd, water_vega], ignore_index=True)
sediment_combined = pd.concat([sediment_hsbd, sediment_vega], ignore_index=True)
print(f"soil_combined: {len(soil_combined)} rows")
print(f"water_combined: {len(water_combined)} rows")
print(f"sediment_combined: {len(sediment_combined)} rows")

soil_combined: 898 rows
water_combined: 896 rows
sediment_combined: 568 rows


In [6]:
def deduplicate_and_count_sources(df: pd.DataFrame, smiles_col: str = "Canonical_smiles") -> pd.DataFrame:
    counts = df.groupby(smiles_col).size().rename("n_sources")
    df = df.drop_duplicates(subset=smiles_col).merge(counts, on=smiles_col)
    return df


soil_before_dedup = len(soil_combined)
water_before_dedup = len(water_combined)
sediment_before_dedup = len(sediment_combined)
soil_data = deduplicate_and_count_sources(soil_combined).reset_index(drop=True)
water_data = deduplicate_and_count_sources(water_combined).reset_index(drop=True)
sediment_data = deduplicate_and_count_sources(sediment_combined).reset_index(drop=True)
print(f"Soil data: {soil_before_dedup} entries before deduplication, {len(soil_data)} after deduplication.")
print(f"Water data: {water_before_dedup} entries before deduplication, {len(water_data)} after deduplication.")
print(f"Sediment data: {sediment_before_dedup} entries before deduplication, {len(sediment_data)} after deduplication.")

Soil data: 898 entries before deduplication, 743 after deduplication.
Water data: 896 entries before deduplication, 743 after deduplication.
Sediment data: 568 entries before deduplication, 414 after deduplication.


In [7]:
# # for visual inspection, optional
# soil_data = soil_data.sort_values("reference")
# water_data = water_data.sort_values("reference")
# sediment_data = sediment_data.sort_values("reference")
# soil_data = soil_data.reset_index(drop=True)
# water_data = water_data.reset_index(drop=True)
# sediment_data = sediment_data.reset_index(drop=True)
# soil_data.to_csv(data_dir / "combined_soil_data.csv", index=False, sep="\t")
# water_data.to_csv(data_dir / "combined_water_data.csv", index=False, sep="\t")
# sediment_data.to_csv(data_dir / "combined_sediment_data.csv", index=False, sep="\t")

In [8]:
# write filtered datasets to db_combined
session_combined = Session_combined()
session_combined.bulk_save_objects(
    [
        SoilData(**{c.name: getattr(row, c.name) for c in SoilData.__table__.columns if c.name != "id"})
        for row in soil_data.itertuples(index=False)
    ]
)
session_combined.bulk_save_objects(
    [
        WaterData(**{c.name: getattr(row, c.name) for c in WaterData.__table__.columns if c.name != "id"})
        for row in water_data.itertuples(index=False)
    ]
)
session_combined.bulk_save_objects(
    [
        SedimentData(**{c.name: getattr(row, c.name) for c in SedimentData.__table__.columns if c.name != "id"})
        for row in sediment_data.itertuples(index=False)
    ]
)
session_combined.commit()
session_combined.close()
print(f"Written to db_combined: {len(soil_data)} soil, {len(water_data)} water, {len(sediment_data)} sediment rows")

Written to db_combined: 743 soil, 743 water, 414 sediment rows
